# 11 — Learning Strategies: Contrastive, Zero-Shot and Multimodal (CLIP)

This notebook connects the project with the learning strategies discussed in class — **self-supervised, contrastive, zero-shot and in-context learning**. It uses CLIP, a model trained contrastively on image–text pairs, to classify H&M garments **without training on our labels**, by comparing image embeddings with text embeddings of prompts such as "a product photo of a pair of trousers".

The zero-shot results are compared with the supervised models, and CLIP image embeddings are also used as frozen features for a linear probe. As an optional extension, the product description `detail_desc` in `articles.csv` provides real image–text pairs to study text-based and multimodal classification.

> **Status: skeleton.** Only the introduction, objectives and work preview are written. Code and results will be added when the phase is developed.

> **Estado de ejecución verificado · 20/11/2026.** Plan de trabajo: todavía sin implementación ni resultados de entrenamiento. Las ablaciones de redes pendientes no se presentan como realizadas.

## Vía C · Embeddings, NLP opcional y conexión con GenAI

Esta es nuestra interpretación del bloque de apuntes «word-vector (embedding) → CNN / RNN / Transformer». **Un embedding es una representación vectorial; por sí solo no es GenAI.** Un vector por palabra de [Word2Vec](https://arxiv.org/abs/1301.3781) es una representación estática. Un encoder contextual como [BERT](https://arxiv.org/abs/1810.04805) produce representaciones dependientes del contexto. Ambos pueden alimentar tareas de clasificación sin generar texto.

| Rama didáctica posible sobre `detail_desc` | Relaciones que aprende | Extensión que se podría contrastar |
|---|---|---|
| Tokenización → embedding → pooling/Dense | Resumen de los vectores | Embeddings aleatorios entrenables frente a preentrenados |
| Tokenización → embedding → CNN1D | Patrones locales de tokens | CNN1D sin/con atención añadida |
| Tokenización → embedding → LSTM/GRU | Dependencias secuenciales mediante estados | Último estado/pooling frente a atención sobre estados |
| Tokens + posición → Transformer encoder | Relaciones mediante self-attention | Encoder pequeño frente a encoder preentrenado |

Esta tabla describe **alternativas opcionales**, no cuatro experimentos ya implementados ni un cambio del objetivo visual principal. Un embedding se adapta a la arquitectura; las palabras no pasan a ser píxeles. La imagen puede representarse como secuencia de columnas (06) o de parches (08), con un significado distinto del token textual.

**Conexión multimodal ya prevista.** [CLIP](https://arxiv.org/abs/2103.00020) aprende a alinear representaciones de imágenes y textos con un objetivo contrastivo. Comparar un embedding de imagen con descripciones de clases permite clasificación zero-shot; no equivale a generar texto o imágenes. Es más preciso describir su origen como preentrenamiento contrastivo con supervisión de pares imagen–texto que tratar todo modelo preentrenado como autosupervisado sin matices. Contraste, transferencia, zero-shot e in-context learning son conceptos relacionados, pero usar un prompt de CLIP no constituye por sí solo un experimento de in-context learning.

**Protocolo si se desarrolla NLP.** Heredar los grupos y splits de 02; ajustar vocabulario propio, longitudes y selección de hiperparámetros con train/validation. Los embeddings externos tendrán su procedencia documentada. Usar solo `detail_desc` como entrada textual, sin añadir la columna objetivo; auditar términos que revelen directamente la categoría. Una descripción puede nombrar la prenda: comparar texto con fotografía como si fuera la misma información sería engañoso. Documentar esa ventaja y estudiar una ablación de términos explícitos usando reglas fijadas sin test. Registrar descripciones vacías o repetidas; si textos idénticos cruzan splits, revisar la unidad de agrupación y versionar cualquier nueva partición, sin sobrescribir la visual.

**BERT y autorregresión.** BERT original reconstruye tokens enmascarados con contexto bidireccional; no se usará como ejemplo de generación causal de siguiente token. La eventual generación tipo GPT se discute en 12. La lectura de los apuntes es nuestra; para confirmar el sentido exacto de la flecha haría falta contrastarla con la explicación de clase.


## Objectives

- Explain the contrastive image–text objective of CLIP and its relation to self-supervised learning.
- Run zero-shot classification of the garment classes with prompt templates and measure validation accuracy.
- Study prompt engineering: class-name wording, templates and prompt ensembling.
- Train a linear probe on frozen CLIP embeddings and compare with transfer learning (notebook 07).
- Optionally test image ↔ text retrieval using `detail_desc` descriptions.
- Draw a diagram of the relationships between the learning strategies for the PRDL report.

## Preview of the work

1. **Concepts.** transfer, self-supervised, contrastive, zero-shot and in-context learning; how they relate.
2. **Loading CLIP.** model, image preprocessing and tokenizer.
3. **Zero-shot classification.** prompts per class, cosine similarity, accuracy and confusion matrix.
4. **Prompt engineering.** effect of templates and class names.
5. **Linear probe on CLIP features.** few-shot and full-data comparison.
6. **Optional multimodal extension.** retrieval and classification with `detail_desc`.
7. **Conclusions.** when zero-shot is enough and when supervised training is still needed.

## Inputs, outputs and course references

| | |
|---|---|
| **Inputs** | Train/validation splits from notebook 02 (test reserved for the final protocol), `articles.csv` (`detail_desc`) |
| **Outputs** | Zero-shot and linear-probe metrics, embedding figures, learning-strategy diagram |
| **Course material** | `Interacting_with_CLIP.ipynb`, `Notebooks_GuideLines_for_DeepLearnig_Report.pptx` (learning-strategy takeaways), `MLLB_HuggingFace_Intro.pptx`, `PRDL_MLLB_LLMs_and_Aplications.pptx` |